# LUX Local analysis notebook version 0.2.0 peptide version

In [ ]:
import copy
import pandas as pd
import warnings
from IPython.display import HTML
import os
import json

import dousatsu as ds
from lux_noctis import lux_protter as protter

## User defined parameters

Adjust these parameters to match FragPipe being QCed.

For a basic analysis, these are all only parameters that should be changed in the notebook.

In [ ]:
# The labels for the 2 classes that will be compared for differential abundance
class_label1 = None
class_label2 = None

# The folder with the outpu of the FragPipe analysis
#input_folder = '../FragPipe'
input_folder = '../FragPipe'

config = None
if os.path.exists('../../config.json'):
    with open('../../config.json', 'r')as f:
        config = json.load(f)

uniprot_annotation_filename = None
surfy_filename = None
gaf_filename = None
obo_filename = None
if config is not None:
    # Uniprot annotation for proteins searched by FragPipe.
    uniprot_annotation_filename = os.path.join(
        config["CELESTIAL_ROOT"], 
        "Celestial/Databases", 
        config["JUPYTER"]["UNIPROT_ANNOTATION"]
    )   

    # Surfy annotation file.
    surfy_filename = os.path.join(
        config["CELESTIAL_ROOT"], 
        "Celestial/Databases", 
        config["JUPYTER"]["SURFY"]
    )
    # GO files
    gaf_filename = os.path.join(
        config["CELESTIAL_ROOT"], 
        "Celestial/Databases", 
        config["JUPYTER"]["GAF"]
    )

    obo_filename = os.path.join(
        config["CELESTIAL_ROOT"], 
        "Celestial/Databases", 
        config["JUPYTER"]["OBO"]
    )

# Quantile threshold for removing proteins with low abundance values
quantile_threshold = 0.25

# Minimum number of peptide a protein has to have to be considered
min_peptide_count = 2

# Should the data be median normalized at the protein level?
normalize_protein = False

# Should missing values be imputed at the protein level?
impute_protein = True

# Should the data be filtered based on CVs?
filter_cv = True

# Drop all samples that are not class_label1 or class_label2
reduce_to_labels = True

# List of samples to drop
drop_samples = []

In [ ]:
# If passed by papermill we get a string that needs to be conveted
# to a python list
if isinstance(drop_samples, str):
    drop_samples = json.loads(drop_samples.replace("'", '"'))

## Load data

### Proteomics

In [ ]:
proteomics_dataset = ds.loaders.factory.ProteomicsDatasetFactory.from_fragpipe(
    input_folder=input_folder,
    class_label1=class_label1,
    class_label2=class_label2,
)

### GO

In [ ]:
# Download GO annotations if not already present
if gaf_filename is None:
    if not os.path.exists('./goa_human.gaf'):
        ds.analysis.download_go_files()
    gaf_filename='./goa_human.gaf'
    
if obo_filename is None:
    if not os.path.exists('./go-basic.obo'):
        ds.analysis.download_go_files()
    obo_filename='./go-basic.obo'
    
goa = ds.analysis.Go(
    gaf_filename=gaf_filename,
    obo_filename=obo_filename
)

### Annotations

In [ ]:
# Download Uniprot and Surfy annotations if not already present
if uniprot_annotation_filename is not None:
    uniprot = ds.loaders.UniprotAnnotationLoader(uniprot_annotation_filename)
elif os.path.exists('./uniprot_annotation.tsv'):
    uniprot = ds.loaders.UniprotAnnotationLoader('./uniprot_annotation.tsv')
else:
    uniprot = ds.loaders.UniprotAnnotationLoader(uniprot_annotation_filename='./uniprot_annotation.tsv', download=True)

if surfy_filename is not None:
    surfy = ds.loaders.SurfyLoader(surfy_filename).load_surfy()
elif os.path.exists('./surfy.txt'):
    surfy = ds.loaders.SurfyLoader('./surfy.txt').load_surfy()    
else:
    ds.loaders.SurfyLoader('./surfy.txt').download_surfy()
    surfy = ds.loaders.SurfyLoader('./surfy.txt').load_surfy()

## Peptide level pre-processing

Make sure we have the proteins we want to use for normalization in the dataset.

In [ ]:
proteomics_dataset.peptide_matrix = (proteomics_dataset.peptide_matrix
    .pipe(ds.preprocessing.drop_samples, samples=drop_samples)
    .pipe(ds.preprocessing.filter_by_quantile, cutoff=quantile_threshold)
    .pipe(ds.preprocessing.filter_best_n_peptides_per_protein)
    .pipe(ds.preprocessing.filter_by_feature_missingness, cutoff=0.66)
    .pipe(ds.preprocessing.filter_peptides_by_min_count, cutoff=min_peptide_count)
    .pipe(ds.preprocessing.normalize_median)
                                    )

In [ ]:
proteomics_dataset.protein_matrix = (proteomics_dataset.peptide_matrix
    .pipe(ds.preprocessing.aggregate_peptides_to_proteins, 
          feature_list=proteomics_dataset.protein_matrix.index.to_list()
         )
)

In [ ]:
if reduce_to_labels:
    proteomics_dataset.reduce_to_classes()

## Overview of protein identifications across runs

We start off by looking at the distribution of protein identifications across samples and classes.

### Total protein identifications per sample

**What to look for:**  
- In LuxLocal the bulk of protein identifications is composed of contaminants. 
Therefore we expect little variation across the classes.

In [ ]:
(ds.visualization
    .plot_feature_per_sample(
        *proteomics_dataset.get_feature_meta_dfs(), 
        lut=proteomics_dataset.classes_palette))

### Distribution of protein identifications across samples

**NOTA BENE**: only classes with at least 10 protein ids are shown

**What to look for:**  
- In LuxLocal the bulk of protein identifications is composed of contaminants. Therefore we expect little variation across the samples.

In [ ]:
with warnings.catch_warnings():
    # Suppress future warnings for external code
    warnings.simplefilter("ignore", category=FutureWarning)
    (ds.visualization
        .plot_upset_features(
            proteomics_dataset.protein_matrix, 
            10))

## Overview of protein abundances across runs

This is were start looking at the distributions of protein abundances across classes/samples.

### Unnormalized protein abundances

**What to look for:**  
- In LuxLocal the bulk of protein identifications is composed of contaminants. Therefore we expect similar distributions and equal medians.

In [ ]:
(ds.visualization
    .plot_intensity_distributions_across_samples(
        feature_matrix=proteomics_dataset.protein_matrix, 
        experiment_annotation=proteomics_dataset.annotation["class"], 
        ylabel=proteomics_dataset.intensity_label,
        lut=proteomics_dataset.classes_palette)
)

### Median normalized protein abundances

If the prior plot show unequal medians we can median normalize protein abundances.

**What to look for:** 

- After median normalization the abundance distributions should be very similar to each other.
- Large divergence in the medians after normalization could indicate that a sample has low intensity (quantile_threshold) or poor coverage (min_peptide_count).

If this is not the case, more complex normalization/batch correction might be required.

In [ ]:
if normalize_protein:
    proteomics_dataset.protein_matrix = (
        proteomics_dataset.protein_matrix.pipe(
            ds.preprocessing.normalize_median
        )
    )
    
    ds.visualization.plot_intensity_distributions_across_samples(
        feature_matrix=proteomics_dataset.protein_matrix,
        y=proteomics_dataset.annotation["class"],
        ylabel=proteomics_dataset.intensity_label,
        lut=meta.classes_palette
    )

**NOTE** Using lux.normalize_protein_abundance_global() woudl have introduced differences.

The lux_noctis implementation of global media normalization is not the same as the one in dousatsu. Specifically, the lux_noctis implementation temporarily sets the most relaxed parameter for the calculation (e.g. quantile_threshold=0; min_peptide_count=0) which is not the case in dousatsu.


### Imputed

If desired missig values can be imputed.

Note, that a very simple imputation is used. Nans are filled with the minimum intensity measured in the corresponding sample minus a random percentage (between 1 and 10%) of said value.

Note, imputing is probably a bad idea and should not be used without a good reason.

In [ ]:
ds.visualization.plot_missingness(proteomics_dataset.protein_matrix, 
                                       mode='sample', 
                                       title="Missingness at the sample level before imputation")

In [ ]:
ds.visualization.plot_missingness(proteomics_dataset.protein_matrix, 
                                       mode='feature', 
                                       title="Missingness at the feature level before imputation")

In [ ]:
if impute_protein:    
    proteomics_dataset.protein_matrix = (
        proteomics_dataset.protein_matrix.pipe(
            ds.preprocessing.impute_nan_with_min,
            randomize=True
        )
    )

    ds.visualization.plot_intensity_distributions_across_samples(
        feature_matrix=proteomics_dataset.protein_matrix,
        experiment_annotation=proteomics_dataset.annotation,
        ylabel=proteomics_dataset.intensity_label,
        lut=proteomics_dataset.classes_palette
    )

## Coefficient of variation

Next we have a look at the coefficient of variation across the classes.

**What to look for:**  
- In label free quantification we are normally looking for CVs below 20%

### Initial

In [ ]:
ds.visualization.plot_cv(*proteomics_dataset.get_feature_meta_dfs())


### Filtered

If desired the dataset can be filtered to remove proteins with high CVs.

By default all proteins with CV higher than 20 in all classes are removed.

In [ ]:
if filter_cv:
    proteomics_dataset.protein_matrix = (
        ds.preprocessing.filter_by_cv(
            feature_matrix=proteomics_dataset.protein_matrix,
            y=proteomics_dataset.annotation,
        )
    )
    ds.visualization.plot_cv(*proteomics_dataset.get_feature_meta_dfs())

## Overview of sample similarities

Next we have a look at sample similarities across classes.

### Intensity based clustering

First we test a simple clustering by protein intensity.

**What to look for:**  

- We would like to see samples from the same class clustering together.
- We should also make sure that we don't have too many missing values.

In [ ]:
(ds.visualization
    .plot_abundance_clustermap(
        *proteomics_dataset.get_feature_meta_dfs(fillna=True), 
        title=proteomics_dataset.intensity_label, 
        lut=proteomics_dataset.classes_palette)
)

### Correlation based clustering

Second we test a correlation based clustering.

**What to look for:**  

- We would like to see samples from the same class clustering together.

In [ ]:
(ds.visualization
    .plot_corr_clustermap(
        feature_matrix=proteomics_dataset.protein_matrix,
        #y=pd.DataFrame(proteomics_dataset.annotation["class"]),
        experiment_annotation=proteomics_dataset.annotation,
        figsize=(10,10),
        lut=proteomics_dataset.classes_palette)
)

Our more stringent protein filtering (due to peptides being counted differently) give much better correlations.

### PCA clustering

Finally we try to cluster the samples by principla component analysis.

**What to look for:**  

- We would like samples from the same class clustering together.

In [ ]:
(ds.visualization
    .plot_pca(
        *proteomics_dataset.get_feature_meta_dfs(fillna=True), 
        plot_stripped=False, 
        lut=proteomics_dataset.classes_palette)
)

## Differential abundance

In this section we compare protein abundances in desired classes.

### Bayes moderated t-test

We implement the bayes moderated t-test workflow with Benjamini-Hochberg correction from limma.

**NOTA BENE**

Even though we are only interested in 2 classes, if additional samples are present, these are kept in the analysis to increase statistical power.  
If this is not desired behaviour, adjust the third parameter (class_labels) of limma_ttest().

In [ ]:
limma_ttest = (
    ds.analysis
        .LimmaTTest(
            protein_matrix=proteomics_dataset.protein_matrix,
            peptide_matrix=proteomics_dataset.peptide_matrix,
            experiment_annotation=proteomics_dataset.annotation,
            class_label1=proteomics_dataset.class_label1,
            class_label2=proteomics_dataset.class_label2,
            is_log_transformed=proteomics_dataset.is_log_transformed)
)

Save:
- top-table
- class1 singularities
- peptides associated with significantly up or class1 singularity proteins

to file.

In [ ]:
limma_ttest.top_table.to_csv('top-table_{}_vs_{}.csv'.format(class_label1, class_label2), index=False)
limma_ttest.singularities_class1.to_csv('singularities_{}.csv'.format(class_label1))
limma_ttest.get_significant_peptides(side=1).to_csv('significant_peptides_{}.csv'.format(class_label1))

### Volcano plot

Visualize the result of the previous analysis in a volcano plot.

**What to look for:**  

- Proteins with significantly different abundance across the tested classes.

Do not over-interpret mean(log2(fold change)).

Be very carefull drawing any conclusion based on identifications that don't meet the significance threshold (i.e. remember that we are only testing to reject the null hypothesis)!

In [ ]:
top_table_annotated = (
    limma_ttest
        .annotate_top_table_with_uniprot(
            uniprot.uniprot_annotation_formatted)
)

(ds.visualization
    .plot_interactive_volcano(
        top_table_annotated, 
        title=f"Volcano plot {proteomics_dataset.class_label1} vs. {proteomics_dataset.class_label2}")
)

In [ ]:
ds.visualization.plot_ma(limma_ttest.top_table)

### Membrane enrichment of differentially abundant proteins

We visualize the membrane enrichment in the significantly changing proteins.

**NOTA BENE**  
By default only up-regulated proteins are considered. If this is not desired behaviour adjust the side parameter of get_membrane_enrichment():
- 0 for both
- 1 for up
- -1 for down

**What to look for:**  

- We would like to see that the majority of significantly enriched proteins are membrane proteins

In [ ]:
uniprot_surface = uniprot.get_cellular_component_proteins()
enrichment_df = pd.DataFrame(
    ds.analysis
        .get_feature_set_enrichment(
            features=limma_ttest
                .get_top_table_selection(side=1)["Entry"],
            feature_set=uniprot_surface,
            set_label="membrane" )
)

enrichment_df = enrichment_df.join(pd.DataFrame(
    ds.analysis
        .get_feature_set_enrichment(
            features=limma_ttest
                .get_top_table_selection(side=1)["Entry"],
            feature_set=surfy,
            set_label="membrane"))).T

enrichment_df

In [ ]:
ds.visualization.plot_feature_enrichment(enrichment_df)

### Intensity distributions of differentially abundant proteins

We plot the intensity of differentially abundant proteins against the overall protein abundance distributions.

**What to look for:**  

- How does the intensity of differentially abundant proteins compare to overall protein abundances? Are they actually enriched in one of the conditions?

NOTE: The legend is sorted by log odds.

In [ ]:
X, y = proteomics_dataset.get_feature_meta_dfs()
ds.visualization.plot_interactive_subset_intensity_over_background(
    limma_ttest.get_significant_proteins(side=1),
    X,
    y,
    uniprot.uniprot_annotation_formatted,
    proteomics_dataset.intensity_label,
    proteomics_dataset.classes_palette
)

### Heatmap visualization of differentially abundant proteins

We visualize the intensity of differentially abundant proteins as a heatmap

In [ ]:
(ds.visualization
    .plot_feature_clustermap(
        limma_ttest.get_significant_proteins(
            side=1, 
            p_cutoff=0.01, 
            fc_cutoff=0),
        proteomics_dataset.intensity_label)
)

## Proteins unique to one class

If a protein is only present in one of the 2 classes, it will be missing in the results of the Bayes moderated t-test. Let's look at them separately.

### Protein abundance distributions

Plot the intensity of proteins unique to one class as a barplot.

**What to look for:**  

- Compare the intensity of  

In [ ]:
ds.visualization.plot_interactive_singularities(
    limma_ttest.singularities_class1,
    proteomics_dataset.class_label1,
    limma_ttest.singularities_class2,
    proteomics_dataset.class_label2,
    uniprot.uniprot_annotation_formatted,
    proteomics_dataset.intensity_label,
    proteomics_dataset.classes_palette)

### Membrane enrichment of unique proteins

We visualize the membrane enrichment in the significantly changing proteins.

**NOTA BENE**  
By default only proteins unique to class_label1 are considered. If this is not desired behaviour adjust the cls parameter of get_membrane_enrichment_singularities():
- 1 for class_label1
- 2 for class_label2

**What to look for:**  

- We would like to see that the majority of significantly enriched proteins are membrane proteins

In [ ]:
uniprot_surface = uniprot.get_cellular_component_proteins()
enrichment_df = pd.DataFrame(
    ds.analysis
        .get_feature_set_enrichment(
            features=limma_ttest
                .singularities_class1.columns,
            feature_set=uniprot_surface,
            set_label="membrane" )
)

enrichment_df = enrichment_df.join(pd.DataFrame(
    ds.analysis
        .get_feature_set_enrichment(
            features=limma_ttest
                .singularities_class1.columns,
            feature_set=surfy,
            set_label="membrane"))).T

enrichment_df

In [ ]:
ds.visualization.plot_feature_enrichment(enrichment_df)

## Gene Ontology Enrichment analysis of differentially abundant proteins

Look for GO term enrichment in the differentially abundant and unique proteins

**What to look for:**  

- Terms associated with plasma membrane or the process being investigated.

In [ ]:
goa.run_go_enrichment_on_top_table(
    limma_ttest.top_table, 
    p_cutoff=0.01, 
    fc_cutoff=1, 
    extra_entries=limma_ttest.singularities_class1.columns)

In [ ]:
if goa.go_results:
    sig_go = goa.go_results_df[
        goa.go_results_df['p_bonferroni'] < 0.05
        ]
    display(sig_go)

**NOTE** There are slight differences in GO enrichment.

However, we already knew that the original version of lux_noctis suffers from a bug that forces any entry that is not matched in the UniProt annotation to disappear from the limma toptable. This explains the different sizes of the populations.

## Peptide level

### Summary peptide table

Generate a table summarizing all peptides for the significant and singularity proteins.

In [ ]:
prtr = protter.Protter(proteomics_dataset, limma_ttest, uniprot)
HTML(prtr.protter_table(highlight_nxst=False))